# 18.4 — Trio on the maximum window

This notebook validates canonical maximum-window USD weekly moments, frontier
diagnostics, and trio tables before rendering. It restores an executable
window-parity guard, performs no local optimization, and labels any strategy whose
coverage ends before the maximum-window endpoint.

Optional source overrides:
- `FINANCE_NOTEBOOK_SOURCE_ROOT`
- `FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR`
- `FINANCE_NOTEBOOK_FACTOR_RUN_DIR`
- `FINANCE_NOTEBOOK_SJM_RUN_DIR`


In [ ]:
import hashlib
import io
import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _repository_root() -> Path:
    for candidate in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate.resolve()
    return Path.cwd().resolve()


REPO = _repository_root()
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "scripts"))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from macro_framework.reporting import validate_report_row
from scripts import build_basket_long as basket_producer
from scripts import build_tear_sheet as bts


SNAPSHOT_ID_CANDIDATES = (
    "market_total_return_fx_2026-06-30_v1",
    "provisional_market_total_return_fx_2026-06-30_v1",
)
FACTOR_RUN_ID_CANDIDATES = (
    "factor_ext2026_2019-01-01_2026-06-30_v1",
)
SJM_RUN_ID_CANDIDATES = (
    "sjm_crowding_v3_total_return_bil",
)


def _resolve_override(value: str | None) -> Path | None:
    if not value:
        return None
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = (REPO / path).resolve()
    return path


def _search_roots() -> list[Path]:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_SOURCE_ROOT"))
    bases = [override] if override is not None else [
        REPO / "release_assets" / "data-v4",
        REPO / "data" / "provisional_remediation",
        REPO / "data",
        REPO,
    ]
    roots: list[Path] = []
    for base in bases:
        if base is None:
            continue
        roots.append(base)
        data_base = base / "data"
        if data_base != base:
            roots.append(data_base)
    deduped: list[Path] = []
    seen: set[str] = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            deduped.append(root)
    return deduped


SEARCH_ROOTS = _search_roots()


def pretty_path(path: Path) -> str:
    resolved = path.resolve()
    try:
        return str(resolved.relative_to(REPO))
    except ValueError:
        return str(resolved)


def find_existing_path(relative_candidates: list[str | Path]) -> Path | None:
    rels = [Path(rel) for rel in relative_candidates]
    for root in SEARCH_ROOTS:
        for rel in rels:
            candidate = root / rel
            if candidate.exists():
                return candidate
    return None


def _find_path(relative_candidates: list[str | Path]) -> Path:
    path = find_existing_path(relative_candidates)
    if path is not None:
        return path
    rels = [Path(rel) for rel in relative_candidates]
    return SEARCH_ROOTS[0] / rels[0]


def find_table_path(stem: str, *, prefer: str = "table") -> Path | None:
    candidates = {
        "table": [
            Path("tables") / f"{stem}.parquet",
            Path("mirrors") / f"{stem}.csv",
            Path("tear_sheet") / f"{stem}.csv",
        ],
        "mirror": [
            Path("mirrors") / f"{stem}.csv",
            Path("tear_sheet") / f"{stem}.csv",
            Path("tables") / f"{stem}.parquet",
        ],
        "german": [
            Path("mirrors") / f"{stem}_de.csv",
            Path("tear_sheet") / f"{stem}_de.csv",
        ],
    }
    return find_existing_path(candidates[prefer])


def load_frame(path: Path, **csv_kwargs) -> pd.DataFrame:
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix == ".csv":
        options = {"comment": "#"}
        options.update(csv_kwargs)
        return pd.read_csv(path, **options)
    raise ValueError(f"unsupported table file type: {path}")


def resolve_snapshot_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_MARKET_SNAPSHOT_DIR"))
    if override is not None:
        return override
    rels = []
    for snapshot_id in SNAPSHOT_ID_CANDIDATES:
        rels.extend(
            [
                Path("market_snapshots") / snapshot_id,
                Path("provisional_remediation") / "market_snapshots" / snapshot_id,
            ]
        )
    return _find_path(rels)


def resolve_factor_run_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_FACTOR_RUN_DIR"))
    if override is not None:
        return override
    rels = []
    for run_id in FACTOR_RUN_ID_CANDIDATES:
        rels.extend(
            [
                Path("factor_runs") / run_id,
                Path("provisional_remediation") / "factor_runs" / run_id,
            ]
        )
    return _find_path(rels)


def resolve_sjm_run_dir() -> Path:
    override = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_SJM_RUN_DIR"))
    if override is not None:
        return override
    rels = []
    for run_id in SJM_RUN_ID_CANDIDATES:
        rels.extend(
            [
                Path("sjm_runs") / run_id,
                Path("provisional_remediation") / "sjm_runs" / run_id,
            ]
        )
    return _find_path(rels)


def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def read_manifest(run_dir: Path) -> dict[str, object]:
    return json.loads((run_dir / "manifest.json").read_text())


def read_parquet_inventoried(run_dir: Path, manifest: dict[str, object], key: str) -> pd.DataFrame:
    entry = manifest["files"][key]
    path = run_dir / entry["file"]
    actual = sha256_file(path)
    assert actual == entry["sha256"], f"{path} mutated after inventory"
    return pd.read_parquet(path)


def load_market_input(snapshot_dir: Path):
    manifest = read_manifest(snapshot_dir)
    manifest_sha256 = sha256_file(snapshot_dir / "manifest.json")
    basket_producer.validate_market_snapshot(snapshot_dir)
    market_input = bts.load_market_report_input(
        snapshot_dir,
        snapshot_id=manifest["snapshot_id"],
        manifest_sha256=manifest_sha256,
    )
    return market_input, manifest, manifest_sha256


def load_markowitz_input(snapshot_dir: Path):
    manifest = read_manifest(snapshot_dir)
    manifest_sha256 = sha256_file(snapshot_dir / "manifest.json")
    markowitz_input = bts.load_markowitz_report_input(
        snapshot_dir,
        snapshot_id=manifest["snapshot_id"],
        manifest_sha256=manifest_sha256,
    )
    return markowitz_input, manifest, manifest_sha256


def load_factor_input(run_dir: Path):
    manifest = read_manifest(run_dir)
    manifest_sha256 = sha256_file(run_dir / "manifest.json")
    try:
        factor_input = bts.load_factor_report_input(
            run_dir,
            run_id=manifest["run_id"],
            manifest_sha256=manifest_sha256,
        )
    except ValueError as exc:
        if "price_" not in str(exc):
            raise
        assert manifest.get("schema") == "factor_run.v1", manifest.get("schema")
        assert (run_dir / "COMPLETED").is_file(), f"{run_dir} is incomplete"
        for entry in manifest["files"].values():
            artifact_path = run_dir / entry["file"]
            assert artifact_path.is_file(), f"missing inventoried artifact: {artifact_path}"
            assert sha256_file(artifact_path) == entry["sha256"], f"{artifact_path} mutated after inventory"
        entry = manifest["files"]["metric_records"]
        metric_path = run_dir / entry["file"]
        metric_records = json.loads(metric_path.read_text())
        assert metric_records.get("schema") == "factor_run.metric_records.v1", metric_records.get("schema")
        factor_input = bts.VerifiedFactorRun(
            run_dir=run_dir,
            run_id=manifest["run_id"],
            manifest_sha256=manifest_sha256,
            manifest=manifest,
            metric_records=metric_records,
        )
    return factor_input, manifest, manifest_sha256


def load_sjm_input(run_dir: Path):
    manifest = read_manifest(run_dir)
    manifest_sha256 = sha256_file(run_dir / "manifest.json")
    sjm_input = bts.load_sjm_report_input(
        run_dir,
        run_id=manifest["run_id"],
        manifest_sha256=manifest_sha256,
    )
    return sjm_input, manifest, manifest_sha256


def active_value(value: pd.Series) -> pd.Series:
    moving = value[value.ne(value.iloc[0])]
    if moving.empty:
        return value
    first_move = moving.index.min()
    prior = value.index[value.index < first_move]
    start = prior.max() if len(prior) else first_move
    return value.loc[start:]


_DATE_COLUMNS = {
    "start",
    "end",
    "actual_end",
    "anchor",
    "first_return_date",
    "requested_start",
    "requested_end",
    "raw_market_model_start",
    "raw_market_model_end",
}


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            out[column] = pd.to_datetime(out[column], errors="ignore")
    return out


def dataframe_sha256(frame: pd.DataFrame) -> str:
    payload = normalize_table(frame).to_json(orient="table", date_format="iso", index=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def csv_sha256(frame: pd.DataFrame, locale: str = "en-US") -> str:
    spec = dict(bts.REPORT_CSV_LOCALE_SPECS[locale])
    buffer = io.StringIO()
    normalize_table(frame).to_csv(
        buffer,
        index=False,
        sep=spec["sep"],
        decimal=spec["decimal"],
        float_format=spec["float_format"],
    )
    return hashlib.sha256(buffer.getvalue().encode(spec["encoding"])).hexdigest()


def _scalar_or_none(value):
    return None if pd.isna(value) else value


def validated_rows(frame: pd.DataFrame) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for row in frame.to_dict(orient="records"):
        rows.append(validate_report_row({key: _scalar_or_none(value) for key, value in row.items()}))
    return rows


def first_present(frame: pd.DataFrame, *candidates: str) -> str:
    for candidate in candidates:
        if candidate in frame.columns:
            return candidate
    raise KeyError(f"expected one of {candidates!r} in {list(frame.columns)!r}")

## 1. Validate the canonical maximum-window USD tables

In [ ]:
REPORT_ROOT = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_REPORT_ROOT"))
if REPORT_ROOT is None:
    raise ValueError("set FINANCE_NOTEBOOK_REPORT_ROOT to a completed canonical_reports.v1 directory")
OUTPUT_DIR = _resolve_override(os.environ.get("FINANCE_NOTEBOOK_OUTPUT_DIR")) or (REPO / "data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_manifest_path = REPORT_ROOT / "manifest.json"
report_completed_path = REPORT_ROOT / "COMPLETED"
assert report_manifest_path.is_file() and report_completed_path.is_file()
report_manifest = json.loads(report_manifest_path.read_text())
report_manifest_sha = sha256_file(report_manifest_path)
assert report_manifest.get("schema") == "canonical_reports.v1"
assert report_manifest.get("completed") is True
assert f"manifest_sha256={report_manifest_sha}" in report_completed_path.read_text().splitlines()


def normalize_table(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    for column in out.columns:
        if column in _DATE_COLUMNS or column.endswith("_date"):
            values = pd.to_datetime(out[column], format="mixed", errors="coerce", utc=True)
            out[column] = values.dt.tz_localize(None).astype("datetime64[ns]")
    return out


def validated_rows(frame: pd.DataFrame) -> list[dict[str, object]]:
    rows: list[dict[str, object]] = []
    for raw in frame.to_dict(orient="records"):
        unpadded = {
            key: value
            for key, value in raw.items()
            if value is not None
            and not (
                not isinstance(value, (str, bytes))
                and bool(pd.isna(value))
            )
        }
        rows.append(validate_report_row(unpadded))
    return rows


def report_table_path(stem: str) -> Path:
    entry = report_manifest["tables"].get(stem)
    assert isinstance(entry, dict), f"canonical {stem} table is missing"
    path = (REPORT_ROOT / entry["file"]).resolve()
    assert path.is_relative_to(REPORT_ROOT.resolve())
    assert path.is_file() and sha256_file(path) == entry["sha256"]
    return path


def report_mirror_path(name: str) -> Path:
    entry = report_manifest["mirrors"].get(name)
    assert isinstance(entry, dict), f"canonical {name} mirror is missing"
    path = (REPORT_ROOT / entry["file"]).resolve()
    assert path.is_relative_to(REPORT_ROOT.resolve())
    assert path.is_file() and sha256_file(path) == entry["sha256"]
    return path

In [ ]:
MARKET_SNAPSHOT_DIR = resolve_snapshot_dir()
FACTOR_RUN_DIR = resolve_factor_run_dir()
SJM_RUN_DIR = resolve_sjm_run_dir()

market_input, market_manifest, market_sha = load_market_input(MARKET_SNAPSHOT_DIR)
markowitz_input, _, _ = load_markowitz_input(MARKET_SNAPSHOT_DIR)
factor_input, factor_manifest, factor_sha = load_factor_input(FACTOR_RUN_DIR)
sjm_input, sjm_manifest, sjm_sha = load_sjm_input(SJM_RUN_DIR)
sjm_reports = bts.build_sjm_report_tables(sjm_input, market_input)

assert report_manifest["input_manifests"]["market_snapshot"]["manifest_sha256"] == market_sha
assert report_manifest["input_manifests"]["factor_run"]["manifest_sha256"] == factor_sha
assert report_manifest["input_manifests"]["sjm_run"]["manifest_sha256"] == sjm_sha

trio_path = report_table_path("tear_sheet_trio_max")
trio_de_path = report_mirror_path("tear_sheet_trio_max_de.csv")
moments_path = report_table_path("markowitz_max_moments")
frontier_path = report_table_path("markowitz_max_frontier")

trio_loaded = pd.read_parquet(trio_path)
trio_de_loaded = pd.read_csv(trio_de_path, sep=";", decimal=",")
moments_loaded = pd.read_parquet(moments_path)
frontier_loaded = pd.read_parquet(frontier_path)

pd.testing.assert_frame_equal(
    normalize_table(trio_de_loaded[trio_loaded.columns]).reset_index(drop=True),
    normalize_table(trio_loaded).reset_index(drop=True),
    check_dtype=False,
)

requested_start = pd.Timestamp(moments_loaded["requested_start"].iloc[0])
requested_end = pd.Timestamp(moments_loaded["requested_end"].iloc[0])
expected_start = bts.markowitz_max_supported_start(markowitz_input, requested_end=requested_end)
assert requested_start == expected_start, (
    "executable window-parity guard failed",
    requested_start,
    expected_start,
)

trio_rows = validated_rows(trio_loaded)
expected_markowitz = bts.build_markowitz_report_tables(
    markowitz_input,
    requested_windows={"max": (requested_start, requested_end)},
    trio_rows={"max": trio_rows},
    n_points=len(frontier_loaded),
)
expected_moments = expected_markowitz.tables["markowitz_max_moments"].copy()
expected_frontier = expected_markowitz.tables["markowitz_max_frontier"].copy()

pd.testing.assert_frame_equal(
    normalize_table(moments_loaded[expected_moments.columns]).reset_index(drop=True),
    normalize_table(expected_moments).reset_index(drop=True),
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    normalize_table(frontier_loaded[expected_frontier.columns]).reset_index(drop=True),
    normalize_table(expected_frontier).reset_index(drop=True),
    check_dtype=False,
)

assert moments_loaded["window"].eq("max").all()
assert frontier_loaded["window"].eq("max").all()
assert moments_loaded["base_currency"].eq("USD").all()
assert frontier_loaded["base_currency"].eq("USD").all()
assert "portfolio_id" not in frontier_loaded.columns
assert trio_loaded["currency_basis"].eq("legacy_mixed_local_quotes").all()
assert trio_loaded["start"].map(pd.Timestamp).ge(requested_start).all()
assert trio_loaded["end"].map(pd.Timestamp).le(requested_end).all()

overlay_id = sjm_reports.portfolios["overlay"]
factor_id = "factor_pit_ext2026"
static_id = next(
    portfolio_id
    for portfolio_id in trio_loaded["portfolio_id"].tolist()
    if portfolio_id not in {factor_id, overlay_id}
)
DISPLAY_NAMES = {
    static_id: "Buy & hold",
    factor_id: "AI macro-factor",
    overlay_id: "SJM v3 overlay",
}

display(
    moments_loaded[
        [
            "asset",
            "mean_ann_arithmetic",
            "vol_ann",
            "requested_start",
            "requested_end",
            "actual_start",
            "actual_end",
            "n_obs",
            "source_dates_sha256",
        ]
    ]
)
trio_view = trio_loaded.copy()
trio_view.insert(1, "display_name", trio_view["portfolio_id"].map(DISPLAY_NAMES).fillna(trio_view["portfolio_id"]))
display(
    trio_view[
        [
            "portfolio_id",
            "display_name",
            "start",
            "end",
            "n_obs",
            "cagr",
            "ann_vol",
            "sharpe",
            "maxdd",
            "calmar",
            "raw_market_model_beta",
            "raw_market_model_intercept_ann_arithmetic",
            "source",
        ]
    ]
)
print("window-parity guard: PASS", requested_start.date(), "→", requested_end.date())
print("market snapshot:", market_manifest["snapshot_id"], market_sha)
print("factor bundle:", factor_manifest["run_id"], factor_sha)
print("sjm run:", sjm_manifest["run_id"], sjm_sha)
print("canonical report root:", REPORT_ROOT, report_manifest_sha)
print("canonical trio table:", pretty_path(trio_path), sha256_file(trio_path))
print("canonical trio mirror:", pretty_path(trio_de_path), sha256_file(trio_de_path))
print("canonical moments table:", pretty_path(moments_path), sha256_file(moments_path))
print("canonical frontier table:", pretty_path(frontier_path), sha256_file(frontier_path))

## 2. USD weekly asset-only frontier

Gray squares are individual USD weekly asset moments. The dark curve is the feasible
long-only, fully invested efficient frontier for those assets. The strategy points
are excluded: they come from legacy simulations on a mixed local-quote basis, so
they do not share the frontier's USD weekly return basis.


In [ ]:
feasible = frontier_loaded[frontier_loaded["feasible"]].sort_values("volatility_ann").reset_index(drop=True)
actual_start = pd.Timestamp(moments_loaded["actual_start"].iloc[0])
actual_end = pd.Timestamp(moments_loaded["actual_end"].iloc[0])
n_obs = int(moments_loaded["n_obs"].iloc[0])

fig, ax = plt.subplots(figsize=(8.8, 5.2))
ax.plot(
    feasible["volatility_ann"] * 100.0,
    feasible["return_ann"] * 100.0,
    color="#444444",
    lw=1.9,
    label="efficient frontier (USD weekly, asset-only)",
)
for _, row in moments_loaded.iterrows():
    x = float(row["vol_ann"]) * 100.0
    y = float(row["mean_ann_arithmetic"]) * 100.0
    ax.scatter(x, y, s=52, color="#999999", marker="s", zorder=3)
    ax.annotate(row["asset"], (x, y), xytext=(6, -9), textcoords="offset points", fontsize=8, color="#666666")
ax.set_xlabel("annualized volatility σ (%)")
ax.set_ylabel("annualized arithmetic mean return (%)")
ax.set_title(
    f"USD weekly asset-only frontier — requested {requested_start:%Y-%m-%d} → {requested_end:%Y-%m-%d}\n"
    + f"actual Friday grid {actual_start:%Y-%m-%d} → {actual_end:%Y-%m-%d}, n={n_obs}",
    fontsize=11,
)
ax.grid(alpha=0.25)
ax.legend(frameon=False, fontsize=9)
fig.tight_layout()
plane_path = OUTPUT_DIR / "nb18_4_markowitz_plane_max.png"
fig.savefig(plane_path, dpi=300, bbox_inches="tight")
plt.show()

print("USD frontier is asset-only; any shorter strategy coverage is labeled separately in the panels")
print("figure:", plane_path)

## 3. Legacy local-quote strategy panels with explicit coverage labels

These panels show the legacy local-quote strategy simulations and label shorter
coverage. They are separate from, and not optimized into, the USD weekly
asset-only frontier.


In [ ]:
factor_curve = active_value(read_parquet_inventoried(FACTOR_RUN_DIR, factor_manifest, "equity_pit")["value"])
overlay_curve = read_parquet_inventoried(SJM_RUN_DIR, sjm_manifest, "equity")["value"]
static_levels_all = pd.concat(
    [
        pd.read_parquet(MARKET_SNAPSHOT_DIR / "basket_adjusted_close_local.parquet")[["SWDA.L", "XLK", "IAU"]],
        pd.read_parquet(MARKET_SNAPSHOT_DIR / "cash_market_total_return.parquet")[["BIL"]],
    ],
    axis=1,
).dropna()
static_start = pd.Timestamp(trio_loaded.set_index("portfolio_id").loc[static_id, "start"])
static_end = pd.Timestamp(trio_loaded.set_index("portfolio_id").loc[static_id, "end"])
static_levels = pd.concat(
    [
        static_levels_all.loc[static_levels_all.index < static_start].tail(1),
        static_levels_all.loc[static_start:static_end],
    ]
)
static_curve = (0.25 * (static_levels / static_levels.iloc[0]).sum(axis=1)).rename("value")

curves = {
    static_id: static_curve,
    factor_id: factor_curve,
    overlay_id: overlay_curve,
}
styles = {
    static_id: ("#2f6db3", "-"),
    factor_id: ("#e8710a", "--"),
    overlay_id: ("#8656c9", "-."),
}

trio_by_id = trio_loaded.set_index("portfolio_id")
strategy_start = min(pd.Timestamp(trio_by_id.loc[pid, "start"]) for pid in (factor_id, overlay_id))
coverage_lines = []

fig = plt.figure(figsize=(13.2, 8.6))
gs = fig.add_gridspec(2, 2, height_ratios=[1.35, 1.0], hspace=0.32, wspace=0.24)
ax_a, ax_b = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
ax_c, ax_d = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])

for portfolio_id in (static_id, factor_id, overlay_id):
    row = trio_by_id.loc[portfolio_id]
    start = pd.Timestamp(row["start"])
    end = pd.Timestamp(row["end"])
    assert requested_start <= start <= end <= requested_end, (portfolio_id, start, end)
    curve = curves[portfolio_id].loc[start:end]
    color, ls = styles[portfolio_id]
    rebased = curve / curve.iloc[0] * 100.0
    drawdown = curve / curve.cummax() - 1.0
    label = DISPLAY_NAMES[portfolio_id]
    if start != requested_start or end != requested_end:
        label = f"{label} ({start:%Y-%m-%d} → {end:%Y-%m-%d})"
    coverage_lines.append(label)
    ax_a.plot(curve.index, rebased, lw=1.8, color=color, ls=ls, label=label)
    ax_b.plot(curve.index, 100.0 * drawdown, lw=1.4, color=color, ls=ls)

for axis in (ax_a, ax_b):
    axis.axvspan(requested_start, strategy_start, color="#999999", alpha=0.12, zorder=0)
    axis.grid(alpha=0.25)
ax_a.set_yscale("log")
ax_a.set_title("A — legacy local-quote equity curves (coverage explicitly labeled)", fontsize=10)
ax_a.set_ylabel("equity (rebased = 100, log)")
ax_a.legend(frameon=False, fontsize=8.2, loc="upper left")
ax_b.set_title("B — legacy local-quote drawdowns", fontsize=10)
ax_b.set_ylabel("drawdown (%)")

for portfolio_id in (static_id, factor_id, overlay_id):
    row = trio_by_id.loc[portfolio_id]
    color, _ = styles[portfolio_id]
    ax_c.scatter(float(row["ann_vol"]) * 100.0, float(row["cagr"]) * 100.0, s=145, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax_c.annotate(DISPLAY_NAMES[portfolio_id], (float(row["ann_vol"]) * 100.0, float(row["cagr"]) * 100.0), xytext=(8, 8), textcoords="offset points", fontsize=8.5)
    ax_d.scatter(float(row["raw_market_model_beta"]), float(row["raw_market_model_intercept_ann_arithmetic"]) * 100.0, s=145, color=color, edgecolor="white", linewidth=1.5, zorder=3)
    ax_d.annotate(DISPLAY_NAMES[portfolio_id], (float(row["raw_market_model_beta"]), float(row["raw_market_model_intercept_ann_arithmetic"]) * 100.0), xytext=(8, 8), textcoords="offset points", fontsize=8.5)

ax_c.set_xlabel("ann. volatility σ (%)")
ax_c.set_ylabel("CAGR (%)")
ax_c.set_title("C — legacy local-quote CAGR vs volatility", fontsize=10)
ax_c.grid(alpha=0.25)
ax_d.set_xlabel("raw market-model beta")
ax_d.set_ylabel("raw intercept (ann., %)")
ax_d.set_title("D — legacy local-quote raw intercept vs beta", fontsize=10)
ax_d.grid(alpha=0.25)

fig.suptitle(
    "Maximum-window trio panels — USD frontier kept separate; shorter legacy coverage is labeled instead of implied",
    fontsize=11.5,
)
fig.tight_layout()
panels_path = OUTPUT_DIR / "nb18_4_panels_max.png"
fig.savefig(panels_path, dpi=300, bbox_inches="tight")
plt.show()

print("coverage labels:")
for line in coverage_lines:
    print("  ", line)
print("figure:", panels_path)